<a href="https://colab.research.google.com/github/furbyhaxx/CLM/blob/main/notebooks/CLM_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning CLM on Colab with Unsloth, without vLLM

A CLM is a frozen **Qwen3-8B** encoder plus two small projection heads (state and action, ~20M
params) trained with InfoNCE. This notebook covers three ways to fine-tune it:

| part | what trains | encoder | data | T4 time |
|---|---|---|---|---|
| 1 | heads | frozen, Unsloth in-process | typed decisions (6k questions) | ~10 min |
| 2 | heads | none, embeddings published | DeepSWE verifier (406k steps) | needs a high-RAM runtime |
| 3 | heads + **LoRA on the encoder** | Unsloth QLoRA | typed decisions | ~30 min |

Parts 1 and 2 are the repo's own `train/finetune.py`. The only change is
`--embed-backend unsloth`, which embeds through `clm.unsloth_embedder` in-process instead of
launching offline vLLM or calling a vLLM server. Part 3 goes further than the repo: it trains a
LoRA adapter on Qwen3-8B jointly with the heads, using Unsloth's gradient checkpointing and
4-bit base weights so that it fits on a free T4.

Heads land in `runs/<name>/best_head.pt`. **CLM_Eval.ipynb** and **CLM_Inference.ipynb** pick
them up from there.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# The CLM repo: the package (heads, Engine, the in-process encoder) and the train / eval scripts.
# It is put on sys.path rather than pip-installed: `contrastive-lm` depends on vLLM, which these
# notebooks do not use.
import os, sys, subprocess
CLM_REPO = "https://github.com/furbyhaxx/CLM"
CLM_REF = "main"
if os.path.isfile(os.path.join("..", "src", "clm", "__init__.py")):   # opened from a clone's notebooks/
    CLM_DIR = os.path.abspath("..")
else:
    CLM_DIR = os.path.abspath("CLM")
    if not os.path.isdir(CLM_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", CLM_REF, CLM_REPO, CLM_DIR], check=True)
for sub in ("src", "train", "preprocessing"):
    sys.path.insert(0, os.path.join(CLM_DIR, sub))
print("CLM:", CLM_DIR)

In [ ]:
# @title GPU
import torch
gpu = torch.cuda.get_device_properties(0)
GPU_GB = gpu.total_memory / 2**30
print(f"GPU = {gpu.name}, {GPU_GB:.1f} GB, bf16 = {torch.cuda.is_bf16_supported()}")

In [ ]:
# @title Settings
from clm.heads import download

DATA, WORKFLOW = "LocalLLaMA/typed-decisions", "all"   # or one workflow, e.g. "customer_service"
INIT_CKPT = download()   # warm start: the reference head Contrastive-LM/CLM-v0.1-8B
EMBED_QUANT = "auto"     # encoder weights: "auto" (4-bit below 20 GB), "4bit" or "16bit"
LOAD_IN_4BIT = {"auto": GPU_GB < 20, "4bit": True, "16bit": False}[EMBED_QUANT]

RUN_DEEPSWE = False      # part 2: 6.2 GB download, ~14 GB of host RAM (Colab Pro high-RAM / A100)
RUN_ENCODER_LORA = True  # part 3
print("encoder in 4-bit:", LOAD_IN_4BIT)

## 1. Typed-decision heads, embedded with Unsloth

Each (row, question) is a state text (context + question) and one candidate text per option. The
target is the annotators' distribution over the options. `finetune.py --task choice`:

1. embeds every unique text once with the in-process Unsloth encoder (length-sorted and
   token-batched; ~6k texts) and caches them in `embeddings/` (the file name records
   backend and quantization, so 4-bit and 16-bit embeddings never mix),
2. trains the heads with bidirectional in-batch InfoNCE over the batch's distinct option texts,
   warm-started from the reference head, and keeps the best validation epoch,
3. reports test accuracy once, against the warm-start head and a majority baseline.

It runs in a subprocess, so the encoder's GPU memory is released when it exits.

In [ ]:
!python {CLM_DIR}/train/finetune.py --task choice --data {DATA} --workflow {WORKFLOW} \
    --init-ckpt {INIT_CKPT} --out-dir runs/typed --embed-cache embeddings \
    --embed-backend unsloth --embed-quant {EMBED_QUANT} --epochs 20

In [ ]:
import json
s = json.load(open("runs/typed/finetune_summary.json"))
print(f"test accuracy: fine-tuned {s['test']['acc']:.3f} | reference head {s['init_test']['acc']:.3f} "
      f"| majority {s['majority_test']:.3f}   (best epoch {s['best_epoch']}, {s['minutes']} min)")
print("per question:", s["test"]["per_question"])

Useful variations: `--loss softce` (softmax over each question's own options), `--targets hard`
(gold labels instead of distributions), `--workflow customer_service` (one domain),
`--epochs` / `--batch` / `--lr`. Embeddings are cached, so re-runs skip the encoder entirely.

## 2. DeepSWE verifier heads (optional)

The DeepSWE step embeddings (8K-token chat-templated states, 406k steps) are published, so
training needs no encoder at all. This reproduces the README's fine-tuned verifier: 75 training
tasks, 38 held out, batch 512, warm-started from the reference head. The embedding store is
6.6 GB in float16 and is loaded into host RAM and GPU memory, so use a high-RAM runtime.

In [ ]:
if RUN_DEEPSWE:
    import psutil
    from huggingface_hub import snapshot_download
    ram = psutil.virtual_memory().total / 2**30
    assert ram > 20, f"{ram:.0f} GB of host RAM; part 2 needs a high-RAM runtime"
    HEADS_DIR = snapshot_download("Contrastive-LM/deepswe-clm-heads-8k", local_dir="heads/deepswe")
    !python {CLM_DIR}/train/finetune.py --task clm --init-ckpt {INIT_CKPT} --out-dir runs/deepswe \
        --holdout-tasks {HEADS_DIR}/heldout_tasks.json --batch 512 --embed-cache embeddings
    # evaluate on the held-out tasks' Opus 5 rollouts (public mirror of the eval embeddings)
    !python {CLM_DIR}/evaluation/bon_eval.py --hf-dataset tarsur385/deepswe-prm-embeddings-8k \
        --checkpoint runs/deepswe/best_head.pt --tasks-file {HEADS_DIR}/heldout_tasks.json \
        --n 4 --window 12
else:
    print("skipped (RUN_DEEPSWE = False)")

For your own agent traces, write step transitions as JSONL (`state` chat messages, `action`,
`task_id`, `step_idx`, `trajectory_id`, `reward`) and pass `--data traces.jsonl
--embed-backend unsloth`. The states are then embedded with the Unsloth encoder using the
training recipe (chat template, last 8191 tokens).

## 3. LoRA on the encoder + heads, with Unsloth (experimental)

Parts 1 and 2 keep Qwen3-8B frozen. Here Unsloth adds LoRA adapters to every projection of the
encoder and trains them **jointly with the heads**, back-propagating through the last-token
pooling. The objective is the soft cross-entropy of each question's option distribution. It
uses Unsloth's usual training setup: 4-bit base weights on a T4, `use_gradient_checkpointing="unsloth"`
and mixed precision.

The result is an encoder of your own. It is saved as a LoRA adapter next to its heads, and both
must be used together: in **CLM_Inference.ipynb** / **CLM_Eval.ipynb**, set
`ENCODER = "runs/typed-lora/encoder_lora"` and load `runs/typed-lora/best_head.pt`.

In [ ]:
# @title LoRA settings
MAX_LEN = 2048          # typed-decision states are at most ~470 tokens
LORA_R = 16
LORA_STEPS = 300        # optimizer steps
LORA_BATCH = 8          # questions per step (each: 1 state + ~3.5 options)
LORA_CHUNK = 32         # sequences per encoder forward
LORA_LR, HEAD_LR = 1e-4, 1e-4
LORA_EVAL_EVERY = 100
LORA_OUT = "runs/typed-lora"
LORA_INIT_HEAD = "runs/typed/best_head.pt" if os.path.exists("runs/typed/best_head.pt") else INIT_CKPT
import time

In [ ]:
if RUN_ENCODER_LORA:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "Qwen/Qwen3-8B",
        max_seq_length = MAX_LEN,
        load_in_4bit = LOAD_IN_4BIT,  # resolved above: 4-bit on a T4
        dtype = None,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = LORA_R,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = LORA_R,
        lora_dropout = 0,  # 0 is optimized
        bias = "none",     # "none" is optimized
        use_gradient_checkpointing = "unsloth",  # 30% less VRAM
        random_state = 3407,
    )

In [ ]:
if RUN_ENCODER_LORA:
    import random
    import adapters
    from finetune import load_typed_rows
    from clm.unsloth_embedder import LocalEncoder, last_token_states

    # the same train / val split as finetune.py --task choice (seed 1234, 10% of train rows)
    train_rows = load_typed_rows(DATA, "train", WORKFLOW, None)
    test_rows = load_typed_rows(DATA, "test", WORKFLOW, None)
    ids = sorted({r["id"] for r in train_rows})
    random.Random(1234).shuffle(ids)
    val_ids = set(ids[:max(1, int(round(0.1 * len(ids))))])
    ex = {"train": list(adapters.typed_decision_examples([r for r in train_rows if r["id"] not in val_ids])),
          "val": list(adapters.typed_decision_examples([r for r in train_rows if r["id"] in val_ids])),
          "test": list(adapters.typed_decision_examples(test_rows))}
    print({k: len(v) for k, v in ex.items()}, "questions")

    # tokenization + pooling of the model being trained (the recipe finetune.py uses: tail, max_len - 1)
    enc = LocalEncoder.from_model(model, tokenizer, max_tokens=MAX_LEN - 1)
    _ids = {}
    def token_ids(text):
        if text not in _ids:
            _ids[text] = enc.text_ids(text, "tail")
        return _ids[text]

In [ ]:
if RUN_ENCODER_LORA:
    import math
    import torch.nn as nn
    import torch.nn.functional as F
    from clm.heads import make_head

    DEVICE = next(p for p in model.parameters()).device
    AMP = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    # heads: warm-started from part 1 (or the reference head), kept in float32
    ck = torch.load(LORA_INIT_HEAD, map_location="cpu", weights_only=False)
    cfg = dict(ck["cfg"])
    head_kw = dict(width=cfg["width"], depth=cfg["depth"], proj=cfg.get("projection_dim", 512),
                   activation=cfg.get("activation", "gelu"), layernorm=cfg.get("layernorm", False),
                   residual=cfg.get("residual", False), hidden=cfg.get("hidden_size", 4096))
    state_head, action_head = make_head(**head_kw).to(DEVICE), make_head(**head_kw).to(DEVICE)
    state_head.load_state_dict(ck["state_head"]); action_head.load_state_dict(ck["action_head"])
    logit_scale = nn.Parameter(torch.as_tensor(ck["logit_scale"]).float().to(DEVICE))

    def encode(texts, chunk=LORA_CHUNK):
        """Unique texts -> (L2-normalised embeddings with grad, {text: row}). Length-sorted chunks pad little."""
        uniq = sorted(dict.fromkeys(texts), key=lambda t: -len(token_ids(t)))
        rows = []
        for i in range(0, len(uniq), chunk):
            part = uniq[i:i + chunk]
            rows.append(F.normalize(last_token_states(model, [token_ids(t) for t in part], enc.pad_id).float(), dim=-1))
        return torch.cat(rows), {t: i for i, t in enumerate(uniq)}

    def question_logits(batch, emb, row):
        """Per question: scale * cos(state_head(state), action_head(option)) over its own options."""
        zs = F.normalize(state_head(emb[[row[e.state_text] for e in batch]]), dim=-1)
        scale = logit_scale.exp().clamp(max=100.0)
        return [scale * F.normalize(action_head(emb[[row[c] for c in e.candidates]]), dim=-1) @ zs[i]
                for i, e in enumerate(batch)]

    def batch_loss(batch):
        """Soft cross-entropy against the annotators' distribution (finetune.py --loss softce --targets soft)."""
        emb, row = encode([t for e in batch for t in (e.state_text, *e.candidates)])
        return sum(-(torch.tensor(e.target, device=DEVICE) * F.log_softmax(lg.float(), -1)).sum()
                   for e, lg in zip(batch, question_logits(batch, emb, row))) / len(batch)

    @torch.no_grad()
    def evaluate(examples):
        FastLanguageModel.for_inference(model); state_head.eval(); action_head.eval()
        texts = list(dict.fromkeys(t for e in examples for t in (e.state_text, *e.candidates)))
        vecs = enc.embed_ids([token_ids(t) for t in texts])          # token-batched, no grad
        emb, row = torch.from_numpy(vecs).to(DEVICE), {t: i for i, t in enumerate(texts)}
        hit, ce = 0, 0.0
        for i in range(0, len(examples), 256):
            batch = examples[i:i + 256]
            for e, lg in zip(batch, question_logits(batch, emb, row)):
                hit += int(lg.argmax()) == e.label
                ce -= float((torch.tensor(e.target, device=DEVICE) * F.log_softmax(lg.float(), -1)).sum())
        FastLanguageModel.for_training(model); state_head.train(); action_head.train()
        return {"acc": hit / len(examples), "soft_ce": ce / len(examples)}

In [ ]:
if RUN_ENCODER_LORA:
    from transformers import get_cosine_schedule_with_warmup

    lora_params = [p for p in model.parameters() if p.requires_grad]
    head_params = list(state_head.parameters()) + list(action_head.parameters()) + [logit_scale]
    opt = torch.optim.AdamW([{"params": lora_params, "lr": LORA_LR}, {"params": head_params, "lr": HEAD_LR}],
                            weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, max(1, LORA_STEPS // 20), LORA_STEPS)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP == torch.float16)
    os.makedirs(LORA_OUT, exist_ok=True)

    def save(tag, metrics):
        model.save_pretrained(os.path.join(LORA_OUT, "encoder_lora"))
        tokenizer.save_pretrained(os.path.join(LORA_OUT, "encoder_lora"))
        torch.save({"state_head": state_head.state_dict(), "action_head": action_head.state_dict(),
                    "logit_scale": logit_scale.detach().cpu(),
                    "cfg": {**cfg, "projection_dim": head_kw["proj"], "hidden_size": head_kw["hidden"],
                            "task": "choice", "encoder": "Qwen/Qwen3-8B", "encoder_lora": "encoder_lora",
                            "encoder_4bit": LOAD_IN_4BIT, "max_len": MAX_LEN},
                    "step": tag, "metrics": metrics}, os.path.join(LORA_OUT, "best_head.pt"))

    FastLanguageModel.for_training(model)
    best = evaluate(ex["val"])
    print(f"step 0 (frozen encoder, part-1 head) val {best}")
    save(0, best)
    rng, t0, running = random.Random(3407), time.time(), 0.0
    for step in range(1, LORA_STEPS + 1):
        with torch.autocast(DEVICE.type, dtype=AMP):
            loss = batch_loss(rng.sample(ex["train"], LORA_BATCH))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(lora_params + head_params, 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        running += loss.item()
        if step % 10 == 0:
            print(f"step {step} loss {running / 10:.4f} ({(time.time() - t0) / step:.2f}s/step)", flush=True)
            running = 0.0
        if step % LORA_EVAL_EVERY == 0 or step == LORA_STEPS:
            m = evaluate(ex["val"])
            print(f"step {step} val {m}", flush=True)
            if m["acc"] > best["acc"]:
                best = m
                save(step, m)
    print("best val", best)

In [ ]:
if RUN_ENCODER_LORA:
    # test once, with the best checkpoint (the adapter and heads saved together)
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    set_peft_model_state_dict(model, load_file(os.path.join(LORA_OUT, "encoder_lora", "adapter_model.safetensors")))
    best_ck = torch.load(os.path.join(LORA_OUT, "best_head.pt"), map_location="cpu", weights_only=False)
    state_head.load_state_dict(best_ck["state_head"]); action_head.load_state_dict(best_ck["action_head"])
    with torch.no_grad():
        logit_scale.copy_(best_ck["logit_scale"].to(DEVICE))
    lora_test = evaluate(ex["test"])
    print(f"LoRA encoder + heads: TEST {lora_test} (best step {best_ck['step']})")

In [ ]:
# @title Compare
if RUN_ENCODER_LORA:
    s = json.load(open("runs/typed/finetune_summary.json")) if os.path.exists("runs/typed/finetune_summary.json") else None
    if s:
        print(f"reference head (zero-shot)     test acc {s['init_test']['acc']:.3f}")
        print(f"part 1: heads, frozen encoder  test acc {s['test']['acc']:.3f}")
    print(f"part 3: heads + LoRA encoder   test acc {lora_test['acc']:.3f}")
    print(f"      (4-bit training encoder: {LOAD_IN_4BIT})")

## Save and share

Heads are plain `torch.save` checkpoints in the format `clm-serve --ckpt` serves. The LoRA adapter
is a standard PEFT directory, loadable by `FastLanguageModel.from_pretrained` and therefore by
`UnslothEmbedder(ENCODER)`. To push both to the Hugging Face Hub:

In [ ]:
if False:  # set True and fill in your repo + token
    from huggingface_hub import HfApi
    api, repo = HfApi(token="YOUR_HF_TOKEN"), "your_name/clm-typed-heads"
    api.create_repo(repo, exist_ok=True)
    api.upload_file(path_or_fileobj="runs/typed/best_head.pt", path_in_repo="typed_head.pt", repo_id=repo)
    if os.path.isdir("runs/typed-lora"):
        api.upload_folder(folder_path="runs/typed-lora", path_in_repo="lora", repo_id=repo)

Serve a head anywhere with `clm-serve --ckpt runs/typed/best_head.pt --local-encoder` (no vLLM),
or evaluate it with **CLM_Eval.ipynb**.